# Mesh Straightening Comparison: Bishop Frame vs. Blender Armature Skinning

This notebook demonstrates and compares two backends for mesh straightening:
1. **Bishop Frame NumPy Backend** (default): A mathematically exact, volume-preserving coordinate projection mapping. It does not suffer from linear interpolation artifacts.
2. **Blender Armature (`bpy`) Backend**: deforms the mesh using Blender's Armature modifier (Linear Blend Skinning / LBS) posed along relative rotations. It demonstrates the classic joint collapse (candy-wrapper distortion) at extreme bends.

We generate a curved cylinder mesh, straighten it with both backends, compute geometric metrics, and display them.

In [ ]:
# Consolidated Imports
import sys
sys.path.append("..")

import numpy as np
import torch
import trimesh
from shapely.geometry import Point

from animgen.animation.straight import straighten
from animgen.core.spline import Spline


In [ ]:
# Step 1: Create a highly curved spline and sweep a cylinder along it
control_points = [
    [0.0, 0.0, 0.0],
    [0.5, 0.2, 1.0],
    [1.0, 0.8, 2.0],
    [0.8, 1.5, 3.0],
    [0.0, 1.8, 4.0],
    [-0.8, 1.2, 5.0],
    [-1.0, 0.0, 6.0],
]
pts = [torch.tensor(pt, dtype=torch.float32) for pt in control_points]
spline = Spline(pts, alpha=0.5)
curve_tensors = spline.evaluate_curve(num_points_per_segment=15)
curve_pts = np.array([t.detach().cpu().numpy() for t in curve_tensors])

radius = 0.3
circle_poly = Point(0, 0).buffer(radius, quad_segs=8)
curved_mesh = trimesh.creation.sweep_polygon(circle_poly, curve_pts)
print(f"Generated curved tube mesh with {len(curved_mesh.vertices)} vertices and {len(curved_mesh.faces)} faces.")
print("Showcasing curved mesh:")
curved_mesh.show()


In [ ]:
# Step 2: Straighten using NumPy (Bishop Frame)
print("Straightening using NumPy (Bishop Frame) backend...")
straightened_numpy = straighten(curved_mesh, spine_points=spline, axis="z", backend="numpy")
print("NumPy straightening completed successfully!")


In [ ]:
# Step 3: Calculate metrics for NumPy backend
dists_xy_numpy = np.linalg.norm(straightened_numpy.vertices[:, :2], axis=1)
print("NumPy Straightening Stats (Expected: perfect cylinder of radius 0.3):")
print(f"  Target radius:               {radius}")
print(f"  Max radial distance:         {np.max(dists_xy_numpy):.4f}")
print(f"  Mean radial distance:        {np.mean(dists_xy_numpy):.4f}")
print(f"  Min radial distance:         {np.min(dists_xy_numpy):.4f}")
print("Showcasing NumPy straightened mesh:")
straightened_numpy.show()


In [ ]:
# Step 4: Straighten using Blender Armature (bpy)
print("Straightening using Blender Armature (bpy) backend...")
straightened_bpy = straighten(curved_mesh, spine_points=spline, axis="z", backend="bpy")
print("bpy straightening completed successfully!")


In [ ]:
# Step 5: Calculate metrics for bpy backend
dists_xy_bpy = np.linalg.norm(straightened_bpy.vertices[:, :2], axis=1)
print("Blender bpy Straightening Stats (Expected: collapsed volume/radius at curved joints):")
print(f"  Target radius:               {radius}")
print(f"  Max radial distance:         {np.max(dists_xy_bpy):.4f}")
print(f"  Mean radial distance:        {np.mean(dists_xy_bpy):.4f}")
print(f"  Min radial distance:         {np.min(dists_xy_bpy):.4f} (collapses toward 0 due to LBS/DQS!)")
print("Showcasing bpy straightened mesh:")
straightened_bpy.show()


In [ ]:
# Step 6: Comparison Summary
print("Comparison Summary:")
print(f"  NumPy Mean Radius:           {np.mean(dists_xy_numpy):.4f} (Ideal: 0.3000)")
print(f"  Blender bpy Mean Radius:      {np.mean(dists_xy_bpy):.4f} (Ideal: 0.3000)")
print(f"\n  NumPy Min Radius:            {np.min(dists_xy_numpy):.4f} (Ideal: 0.3000)")
print(f"  Blender bpy Min Radius:       {np.min(dists_xy_bpy):.4f} (Ideal: 0.3000)")
